In [ ]:
import pandas as pd
import json

# Load the JSON file
with open("datasets/er_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)

# Select an id (example: 'wikipedia_000000')
selected_id = "wikipedia_000000"
selected = df[df["id"] == selected_id][["text", "mentions"]]

selected

,text,mentions
0,Anarchism is a political philosophy and moveme...,"[{'mention': 'William Godwin', 'start': 2658, ..."


In [ ]:
import pandas as pd
import json
from IPython.display import HTML, display

# Load the JSON file
with open("datasets/er_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)

# Select an id (example: 'wikipedia_000000')
selected_id = "wikipedia_000003"
row = df[df["id"] == selected_id].iloc[0]

text = row.get("text", "")
mentions = row.get("mentions", [])

# Highlight mentions in text with tooltip for canonical_name and description
import html

highlighted_text = text
if mentions:
    # Sort mentions by start index, then by longest span (end-start) descending
    mentions_sorted = sorted(
        mentions, key=lambda m: (m["start"], -(m["end"] - m["start"]))
    )
    # Track covered indices to avoid overlaps
    covered = set()
    for mention in mentions_sorted[::-1]:  # reverse for safe string replacement
        start, end = mention["start"], mention["end"]
        # Skip if any index in this span is already covered
        if any(i in covered for i in range(start, end)):
            continue
        tooltip = (
            f"{mention.get('canonical_name', '')}: {mention.get('description', '')}"
        )
        tooltip_escaped = html.escape(tooltip, quote=True)
        entity_text_escaped = html.escape(highlighted_text[start:end])
        highlighted_text = (
            highlighted_text[:start]
            + f"<span style='color: red; font-weight: bold;' title=\"{tooltip_escaped}\">{entity_text_escaped}</span>"
            + highlighted_text[end:]
        )
        # Mark indices as covered
        covered.update(range(start, end))

display(HTML(highlighted_text))

with open(f"outputs/highlighted_entities_{selected_id}.html", "w", encoding="utf-8") as f:
    f.write(highlighted_text)